# Your API Key and First Model Call

## What you'll learn
- How to get an API key from Anthropic (Claude) or OpenAI (ChatGPT)
- How to set the key safely so it never appears in a notebook
- How to make your first model call from Python
- How to handle the errors you are most likely to see

## Step 1: get an API key

Your code talks to the model through an **API key**, a secret string that identifies your
account. Use whichever provider you have:

| Provider | Where to create a key | Environment variable |
|----------|----------------------|----------------------|
| Anthropic (Claude) | https://console.anthropic.com (API Keys page) | `ANTHROPIC_API_KEY` |
| OpenAI (ChatGPT) | https://platform.openai.com/api-keys | `OPENAI_API_KEY` |

API usage is billed per request, separately from a chat subscription, so make sure your
API account has billing or credits set up. Every lesson here uses small requests.

**Treat the key like a password.** Never paste it into a code cell, never email it, and
never commit it to git. If a key leaks, delete it on the provider's site and create a new one.

## Step 2: set the key before you start Jupyter

The safest place for the key is an **environment variable**: a setting your terminal
passes to programs it starts. Python reads it with `os.environ`, so the key never lives in
the notebook. Close JupyterLab, then in a terminal run the line for your system and
provider, and start `jupyter lab` again **from that same terminal**:

**Mac / Linux** (lasts until the terminal is closed):
```
export ANTHROPIC_API_KEY="paste-your-key-here"
jupyter lab
```

**Windows PowerShell** (lasts until the window is closed):
```
$env:ANTHROPIC_API_KEY = "paste-your-key-here"
jupyter lab
```

For OpenAI, use `OPENAI_API_KEY` instead. To make it permanent: on Mac/Linux add the
`export` line to `~/.zshrc` or `~/.bashrc`; on Windows run
`setx ANTHROPIC_API_KEY "paste-your-key-here"` once and open a new window.

**No terminal access?** The setup cell below asks for the key with a hidden prompt. It is
kept in memory for this session only and is not saved in the notebook.

## Step 3: install the libraries

Both provider libraries are in `requirements.txt`. If you skipped that step:
```
pip install anthropic openai
```

## Step 4: choose your provider and connect

Set `PROVIDER` to `"anthropic"` or `"openai"`, then run the cell. The same cell appears at
the top of every lesson in this stage, so you only ever change that one line.

In [ ]:
import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)

## Step 5: your first call

Each provider has its own way of sending a prompt, but the idea is the same: you send the
model name and the message, and get text back.

- **Anthropic:** `client.messages.create(...)` with a list of `messages`. The reply is a
  list of content blocks; we keep the text ones.
- **OpenAI:** `client.responses.create(...)` with `input`. The reply text is `response.output_text`.

`ask_llm` below wraps both, so the rest of the lesson works with either provider. It also
catches errors and prints a plain explanation instead of a long traceback.

In [ ]:
def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"

In [ ]:
def ask_llm(prompt, system=None):
    """Send one prompt to the chosen provider and return the reply text."""
    if client is None:
        return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                **extra,
            )
            if response.stop_reason == "refusal":
                print("The model declined this request.")
                return None
            # The reply is a list of blocks; keep the text ones.
            return "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(model=MODEL, input=prompt, **extra)
            return response.output_text
    except sdk.APIError as error:
        print(explain_error(error))
        return None

In [ ]:
reply = ask_llm("In two sentences, what does an FP&A team do?")
if reply:
    print(reply)

## Common errors and what they mean

| Message | Usual cause |
|---------|-------------|
| The API key was rejected | Typo in the key, the key was deleted, or it belongs to the other provider |
| Rate limit or usage quota reached | Too many requests too fast, or no billing/credits on the account |
| Could not reach the API | No internet, or a company network or proxy blocking the request |
| `... is not set` | Jupyter was started before the key was set; restart it from the same terminal |

## Cost in one paragraph

Providers charge per **token** (roughly three quarters of a word), for both what you send
and what you get back. A short question and answer costs a tiny fraction of a cent, but a
loop that sends a large table on every step adds up. That is one reason agents send small
tool results instead of whole datasets. Check your provider's pricing page for current rates.

## Practice

1. Ask the model to explain "favourable vs unfavourable variance" in one sentence.
2. Ask the same question twice. Are the answers identical? Why might they differ?
3. Deliberately break the key: set `os.environ[KEY_NAMES[PROVIDER]] = "wrong"`, run
   `client = make_client()`, and call `ask_llm` again. Read the message, then restart the
   kernel to get your real key back.
4. If you have keys for both providers, switch `PROVIDER`, re-run the setup cell, and ask
   the same question.

In [ ]:
# Your practice code here

## Summary

- An API key identifies your account; keep it out of notebooks and git.
- Set it as `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` before starting Jupyter, or use the hidden prompt.
- Anthropic uses `client.messages.create`; OpenAI uses `client.responses.create`.
- Wrap calls in `try` / `except` so errors are readable.
- You pay per token, so send the model only what it needs.

## Practice Solutions

In [ ]:
import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)



def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"



def ask_llm(prompt, system=None):
    """Send one prompt to the chosen provider and return the reply text."""
    if client is None:
        return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                **extra,
            )
            if response.stop_reason == "refusal":
                print("The model declined this request.")
                return None
            # The reply is a list of blocks; keep the text ones.
            return "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(model=MODEL, input=prompt, **extra)
            return response.output_text
    except sdk.APIError as error:
        print(explain_error(error))
        return None


# 1.
print(ask_llm("In one sentence, what is a favourable vs an unfavourable budget variance?"))

# 2. Models sample their words, so repeated answers usually differ slightly in wording.
for _ in range(2):
    print(ask_llm("Give one tip for presenting a budget variance to executives."))

# 3. Uncomment to try, then restart the kernel afterwards.
# os.environ[KEY_NAMES[PROVIDER]] = "wrong"
# client = make_client()
# print(ask_llm("hello"))

# 4. Change PROVIDER at the top of the setup code, re-run, and ask again.